# Numerikus módszerek – 9. hét gyakorlat
## Iteratív LER-módszerek · Nemlineáris egyenletek

**Feladatok:**
- **Papíron számolós feladatok** – minden cella elején leírjuk a megoldás lépéseit,  a Python-kód csak az ellenőrzést végzi.
- **Debuggolós feladatok**

> **Jelmagyarázat:** 📝 = papíron számolós &nbsp;&nbsp; 🐛 = debuggolós

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=6, suppress=True)

---
# 📝 Papíron számolós feladatok


## 1. feladat – Gauss–Seidel lépések kézzel 📝

Legyen
$$A = \begin{bmatrix}4&-1\\-1&4\end{bmatrix}, \quad b = \begin{bmatrix}3\\7\end{bmatrix},
\quad x^{(0)} = \begin{bmatrix}0\\0\end{bmatrix}.$$

**Végezz 2 Gauss–Seidel-iterációs lépést kézzel!**

*Emlékeztető:* Az $i$-edik egyenletből:
$$x_i^{(k+1)} = \frac{1}{a_{ii}}\left(b_i - \sum_{j<i}a_{ij}x_j^{(k+1)} - \sum_{j>i}a_{ij}x_j^{(k)}\right)$$

---
**Megoldás vázlata:**

$k=0 \to k=1$:
$$x_1^{(1)} = \frac{1}{4}(3 - (-1)\cdot 0) = \frac{3}{4} = 0.75$$
$$x_2^{(1)} = \frac{1}{4}(7 - (-1)\cdot 0.75) = \frac{7.75}{4} = 1.9375$$

$k=1 \to k=2$:
$$x_1^{(2)} = \frac{1}{4}(3 + 1.9375) = \frac{4.9375}{4} \approx 1.2344$$
$$x_2^{(2)} = \frac{1}{4}(7 + 1.2344) \approx 2.0586$$

Pontos megoldás: $x^* = [??, ??]$ – ellenőrizd az alábbi kóddal!

In [ ]:
A = np.array([[4,-1],[-1,4]], float)
b = np.array([3,7], float)
x_star = np.linalg.solve(A, b)
print('Pontos megoldás:', x_star)

x = np.zeros(2)
for k in range(2):
    x_new = x.copy()
    for i in range(2):
        sigma = sum(A[i,j]*x_new[j] for j in range(i)) + sum(A[i,j]*x[j] for j in range(i+1,2))
        x_new[i] = (b[i]-sigma)/A[i,i]
    x = x_new
    print(f'k={k+1}: x = {x}')

## 2. feladat – Optimális SOR paraméter meghatározása 📝

Legyen
$$A = \begin{bmatrix}2&-1&0\\-1&2&-1\\0&-1&2\end{bmatrix}.$$

1. Állapítsd meg, hogy alkalmazható-e az optimális SOR-tétel!
2. Számítsd ki $\varrho(B_J)$-t (a Jacobi-iteráció spektrálsugarát)!
3. Add meg az optimális $\omega_0$-t!

---
**Megoldás vázlata:**

$A$ szimmetrikus, pozitív definit és tridiagonális → tétel alkalmazható ✓

$$B_J = D^{-1}(L+U) = \frac{1}{2}\begin{bmatrix}0&1&0\\1&0&1\\0&1&0\end{bmatrix}$$

Sajátértékek: $0, \pm\frac{1}{\sqrt{2}}$, tehát $\varrho(B_J) = \frac{1}{\sqrt{2}}$.

$$\omega_0 = \frac{2}{1+\sqrt{1-\frac{1}{2}}} = \frac{2}{1+\frac{1}{\sqrt{2}}} = 4-2\sqrt{2}\approx 1.1716$$

In [ ]:
A = np.array([[2,-1,0],[-1,2,-1],[0,-1,2]], float)
D = np.diag(np.diag(A))
L = -np.tril(A, -1)
U = -np.triu(A, 1)
BJ = np.linalg.inv(D) @ (L+U)
rho_BJ = max(abs(np.linalg.eigvals(BJ)))
omega0 = 2 / (1 + np.sqrt(1 - rho_BJ**2))
print(f'rho(B_J)  = {rho_BJ:.6f}  (kell: {1/np.sqrt(2):.6f})')
print(f'omega_0   = {omega0:.6f}  (kell: {4-2*np.sqrt(2):.6f})')
print(f'rho(B_S(omega_0)) = {omega0-1:.6f}')

## 3. feladat – SOR-tétel alkalmazhatósága 📝

Döntsd el az alábbi mátrixokról, hogy alkalmazható-e rájuk az optimális SOR-tétel!
Ha igen, add meg $\omega_0$-t.

(a) $A_1 = \begin{bmatrix}3&-1&0\\-1&3&-1\\0&-1&3\end{bmatrix}$

(b) $A_2 = \begin{bmatrix}2&1\\-1&3\end{bmatrix}$

(c) $A_3 = \begin{bmatrix}4&-1&-1&0\\-1&4&0&-1\\-1&0&4&-1\\0&-1&-1&4\end{bmatrix}$

---
*(Megoldás: (a) igen – tridiag, SPD; (b) nem – nem szimmetrikus; (c) igen – SPD, de NEM tridiagonális
→ a tétel csak tridiagonálisra garantál optimumot.)*

In [ ]:
for name, A in [
    ('A1', np.array([[3,-1,0],[-1,3,-1],[0,-1,3.0]])),
    ('A2', np.array([[2.,1],[-1,3]])),
    ('A3', np.array([[4,-1,-1,0],[-1,4,0,-1],[-1,0,4,-1],[0,-1,-1,4.0]]))
]:
    symm = np.allclose(A, A.T)
    pd   = np.all(np.linalg.eigvalsh(A) > 0) if symm else False
    n    = A.shape[0]
    trid = all(A[i,j]==0 for i in range(n) for j in range(n) if abs(i-j)>1)
    print(f'{name}: szimm={symm}, PD={pd}, tridiag={trid}')
    if symm and pd and trid:
        D = np.diag(np.diag(A)); L=-np.tril(A,-1); U=-np.triu(A,1)
        BJ = np.linalg.inv(D)@(L+U)
        rho = max(abs(np.linalg.eigvals(BJ)))
        w0 = 2/(1+np.sqrt(1-rho**2))
        print(f'  -> rho(BJ)={rho:.4f}, omega0={w0:.4f}')

## 4. feladat – Richardson konvergencia tartomány 📝

$$A = \begin{bmatrix}5&1\\1&5\end{bmatrix}, \quad b = \begin{bmatrix}6\\6\end{bmatrix}$$

Add meg azt a $p\in\mathbb{R}$ értéktartományt, amelyre a Richardson-iteráció $R(p)$ konvergens!

---
**Megoldás:**

Sajátértékek: $\lambda_1 = 4$, $\lambda_2 = 6$, tehát $m=4$, $M=6$.

Konvergencia feltétele: $p\in\left(0,\,\dfrac{2}{M}\right) = \left(0,\,\dfrac{1}{3}\right)$.

In [ ]:
A = np.array([[5.,1],[1,5]])
eigs = np.linalg.eigvalsh(A)
m, M = eigs.min(), eigs.max()
print(f'Sajátértékek: m={m}, M={M}')
print(f'Konvergencia: p ∈ (0, {2/M:.4f})')
print(f'Optimális p0 = {2/(M+m):.4f}, q = {(M-m)/(M+m):.4f}')

## 5. feladat – Optimális Richardson paraméter 📝

Az előző feladat mátrixára ($A=\begin{bmatrix}5&1\\1&5\end{bmatrix}$):

1. Add meg az optimális $p_0$-t!
2. Add meg a hozzá tartozó $q$ kontrakciós együtthatót!
3. Hány lépés szükséges $10^{-4}$ pontossághoz, ha $\|x^{(0)}-x^*\|\le 1$?

---
**Megoldás:**

$$p_0 = \frac{2}{M+m} = \frac{2}{6+4} = \frac{1}{5},\qquad
q = \frac{M-m}{M+m} = \frac{6-4}{10} = \frac{1}{5}$$

Hibabecslés: $\|x^{(k)}-x^*\|\le q^k\|x^{(0)}-x^*\|\le (1/5)^k$.

$(1/5)^k < 10^{-4} \Rightarrow k > \dfrac{4\ln 10}{\ln 5} \approx 5.72$, tehát **6 lépés**.

In [ ]:
m, M = 4., 6.
p0 = 2/(M+m)
q  = (M-m)/(M+m)
print(f'p0 = {p0:.4f} = 1/5 = {1/5:.4f}')
print(f'q  = {q:.4f} = 1/5 = {1/5:.4f}')
k_min = int(np.ceil(4*np.log(10)/np.log(5)))
print(f'Szükséges lépések: k > {4*np.log(10)/np.log(5):.2f} -> {k_min} lépés')
print(f'Ellenőrzés: (1/5)^{k_min} = {(1/5)**k_min:.2e} < 1e-4: {(1/5)**k_min<1e-4}')

## 6. feladat – Richardson-lépések kézzel 📝

$$A = \begin{bmatrix}5&1\\1&5\end{bmatrix},\quad b = \begin{bmatrix}6\\6\end{bmatrix},
\quad x^{(0)} = \begin{bmatrix}0\\0\end{bmatrix},\quad p = \frac{1}{5}$$

Végezz 2 Richardson-lépést kézzel (reziduum-vektoros alakban)!

---
**Megoldás:**

$r^{(0)} = b - Ax^{(0)} = [6,6]^T$

$s^{(0)} = p\cdot r^{(0)} = \frac{1}{5}[6,6]^T = [1.2,\,1.2]^T$

$x^{(1)} = x^{(0)} + s^{(0)} = [1.2,\,1.2]^T$

$r^{(1)} = r^{(0)} - As^{(0)} = [6,6]^T - \begin{bmatrix}5&1\\1&5\end{bmatrix}[1.2,1.2]^T = [6,6]^T - [7.2,7.2]^T = [-1.2,-1.2]^T$

$s^{(1)} = \frac{1}{5}[-1.2,-1.2]^T = [-0.24,-0.24]^T$

$x^{(2)} = x^{(1)} + s^{(1)} = [0.96,\,0.96]^T$

In [ ]:
A = np.array([[5.,1],[1,5]])
b = np.array([6.,6])
x_star = np.linalg.solve(A, b)
print('Pontos megoldás:', x_star)

x = np.zeros(2); r = b - A@x; p = 1/5
for k in range(2):
    s = p*r; x = x+s; r = r-A@s
    print(f'k={k+1}: x={x}, ||hiba||={np.linalg.norm(x-x_star):.6f}')

## 7. feladat – Intervallumfelezés kézzel 📝

$$f(x) = x^3 - 2x - 5$$, intervallum: $$[2, 3]$$.

Végezz **3 intervallumfelezési lépést** kézzel! Töltsd ki a táblázatot:

| k | a | b | c=(a+b)/2 | f(c) | új intervallum |
|-----|-----|-----|-------------|--------|----------------|
| 0   | 2   | 3   | 2.5         | −0.375 | [2.5, 3]       |
| 1   | 2.5 | 3   | ...         | ...    | ...            |
| 2   | ... | ... | ...         | ...    | ...            |

*Ellenőrzés:* $f(2)=-1<0$, $f(3)=16>0$ ✓

In [ ]:
f = lambda x: x**3 - 2*x - 5
a, b = 2., 3.
print(f'f(2) = {f(2)}, f(3) = {f(3)}')
print(f'{'k':>2}  {'a':>8}  {'b':>8}  {'c':>8}  {'f(c)':>10}')
for k in range(5):
    c = (a+b)/2
    fc = f(c)
    print(f'{k:>2}  {a:>8.5f}  {b:>8.5f}  {c:>8.5f}  {fc:>10.6f}')
    if f(a)*fc < 0: b = c
    else: a = c

## 8. feladat – Szükséges lépésszám 📝

Hány intervallumfelezési lépés szükséges, hogy az $[1, 2]$ intervallumon $10^{-4}$ pontosságot
elérjünk?

---
**Megoldás:**

Hibabecslés: $|x_k - x^*| \le \dfrac{b-a}{2^k} = \dfrac{1}{2^k}$.

$$\frac{1}{2^k} < 10^{-4} \Rightarrow 2^k > 10^4 \Rightarrow k > \frac{4\ln 10}{\ln 2} \approx 13.29$$

Tehát **14 lépés** szükséges.

In [ ]:
import math
k_min = math.ceil(4*math.log(10)/math.log(2))
print(f'Szükséges lépések: k > {4*math.log(10)/math.log(2):.2f} -> {k_min} lépés')
print(f'Ellenőrzés: 1/2^{k_min} = {1/2**k_min:.2e} < 1e-4: {1/2**k_min < 1e-4}')

## 9. feladat – Bolzano-tétel alkalmazása 📝

Állapítsd meg, van-e gyöke az $f(x) = e^x - 3x^2$ függvénynek a $[0,1]$ intervallumon!
Alkalmazható-e a Bolzano-tétel?

---
**Megoldás:**

$f(0) = e^0 - 0 = 1 > 0$

$f(1) = e - 3 \approx 2.718 - 3 = -0.282 < 0$

$f\in C[0,1]$ ✓, $f(0)\cdot f(1) < 0$ ✓ → Bolzano-tétel alkalmazható → $\exists\, x^*\in(0,1): f(x^*)=0$.

In [ ]:
import math
f = lambda x: math.exp(x) - 3*x**2
print(f'f(0) = {f(0):.4f}')
print(f'f(1) = {f(1):.4f}')
print(f'f(0)*f(1) = {f(0)*f(1):.4f} < 0: {f(0)*f(1)<0} -> Bolzano alkalmazható!')

# gyök keresése
a, b_ = 0., 1.
for _ in range(40):
    c = (a+b_)/2
    if f(a)*f(c) < 0: b_ = c
    else: a = c
print(f'Gyök közelítése: x* ≈ {(a+b_)/2:.8f}, f(x*) ≈ {f((a+b_)/2):.2e}')

## 10. feladat – Kontrakció igazolása 📝

Igazold, hogy $\varphi(x) = \dfrac{\cos(x)}{2} + 1$ kontrakció a $[0, 2]$ intervallumon!
Add meg a kontrakciós együtthatót $q$-t!

---
**Megoldás:**

1. **Leképezi-e $[0,2]$-t önmagára?**
   $\varphi(0) = 1/2+1 = 1.5$, $\varphi(2) = \cos(2)/2+1 \approx 0.792/2+1 = 1.208$.
   $\varphi$ monoton csökkenő $[0,2]$-n (mert $\varphi'(x)=-\sin(x)/2\le 0$).
   Tehát $\varphi([0,2])\approx[1.208,1.5]\subset[0,2]$ ✓

2. **Kontrakció?**
   $|\varphi'(x)| = \dfrac{|\sin(x)|}{2} \le \dfrac{\sin(2)}{2} \approx \dfrac{0.909}{2} \approx 0.455 < 1$

   Tehát $q = \sin(2)/2 \approx 0.455$.

In [ ]:
phi  = lambda x: np.cos(x)/2 + 1
dphi = lambda x: -np.sin(x)/2
xs = np.linspace(0, 2, 1000)
q = np.max(np.abs(dphi(xs)))
print(f'max |phi\'(x)| = sin(2)/2 = {np.sin(2)/2:.6f}')
print(f'phi([0,2]) ∈ [{phi(xs).min():.4f}, {phi(xs).max():.4f}] ⊂ [0,2]')
print(f'Kontrakciós együttható q ≈ {q:.6f} < 1: {q<1}')

# Fixpont keresése iterációval
x = 1.0
for _ in range(50): x = phi(x)
print(f'Fixpont: x* ≈ {x:.8f}, ellenőrzés: phi(x*)={phi(x):.8f}')

## 11. feladat – Banach-tétel feltételei 📝

$\varphi(x) = \dfrac{x^2}{3} + \dfrac{1}{3}$

1. Leképezi-e $[0,1]$-t önmagára?
2. Kontrakció-e?
3. Add meg a fixpontot!
4. Hány lépés kell 0.01-es pontossághoz $x_0 = 0.5$-ből indulva?

---
**Megoldás:**

1. $\varphi(0)=1/3$, $\varphi(1)=2/3$. $\varphi$ monoton növő $[0,1]$-n.
   $\varphi([0,1])=[1/3,2/3]\subset[0,1]$ ✓

2. $|\varphi'(x)| = |2x/3| \le 2/3 < 1$ for $x\in[0,1]$. Igen, $q=2/3$. ✓

3. Fixpont: $x^* = x^{*2}/3+1/3 \Rightarrow 3x^* = x^{*2}+1 \Rightarrow x^{*2}-3x^*+1=0$
   $\Rightarrow x^* = \frac{3-\sqrt{5}}{2}\approx 0.382$ (a $[0,1]$-beli gyök).

4. $|x_k-x^*| \le \dfrac{q^k}{1-q}|x_1-x_0|$.
   $x_1=\varphi(0.5)=1/12+1/3=5/12\approx 0.417$, $|x_1-x_0|=|0.417-0.5|=0.083$.
   Kell: $\dfrac{(2/3)^k}{1/3}\cdot 0.083 < 0.01$.

In [ ]:
phi  = lambda x: x**2/3 + 1/3
dphi = lambda x: 2*x/3
xs = np.linspace(0,1,1000)
q = np.max(np.abs(dphi(xs)))
x_star = (3-np.sqrt(5))/2
print(f'q = max|phi\'| = {q:.4f}')
print(f'Fixpont: x* = (3-sqrt(5))/2 = {x_star:.6f}')
print(f'Ellenőrzés: phi(x*) = {phi(x_star):.6f}')

x = 0.5
x1 = phi(x)
dx = abs(x1-x)
k_min = int(np.ceil(np.log(0.01*(1-q)/dx)/np.log(q)))
print(f'|x1-x0| = {dx:.4f}, szükséges lépések: {k_min}')

# Ellenőrzés iterációval
for k in range(50):
    xn = phi(x)
    if abs(xn-x_star) < 0.01:
        print(f'{k+1} lépés után |x_k - x*| = {abs(xn-x_star):.4f} < 0.01')
        break
    x = xn

## 12. feladat – Konvergencia rend meghatározása 📝

Határozd meg az alábbi nullsorozat konvergencia rendjét!

$$x_k = \frac{1}{3^k}$$

---
**Megoldás:**

Tippeljük $p=1$-re:
$$\lim_{k\to\infty}\frac{|x_{k+1}-0|}{|x_k-0|^1} = \lim_{k\to\infty}\frac{1/3^{k+1}}{1/3^k} = \frac{1}{3} \in (0,1)$$

A konvergencia **elsőrendű** ($p=1$), $c=1/3$.

In [ ]:
k_vals = np.arange(1, 20)
xk   = 1.0/3**k_vals
xk1  = 1.0/3**(k_vals+1)
ratio_p1 = xk1 / xk
ratio_p2 = xk1 / xk**2
print(f'p=1 hányados: {ratio_p1[:5]} -> határ: {ratio_p1[-1]:.6f}')
print(f'p=2 hányados: {ratio_p2[:5]} -> határ: {ratio_p2[-1]:.6e} (nem véges -> nem p=2)')

## 13. feladat – Konvergencia rend Taylor-sorral 📝

Az $x^3-x-1=0$ egyenlet $\varphi(x)=\sqrt[3]{x+1}$ iterációjának konvergencia rendje?

---
**Megoldás (Taylor-sor alapján):**

A fixpont $x^*\approx 1.3247$. Szükséges feltétel a magasabb rendhez: $\varphi'(x^*)=0$?

$$\varphi'(x) = \frac{1}{3}(x+1)^{-2/3}, \quad \varphi'(x^*) = \frac{1}{3}(x^*+1)^{-2/3}\approx 0.259 \ne 0$$

Tehát $\varphi'(x^*)\ne 0$ → a konvergencia **elsőrendű**, $q\approx 0.259$.

In [ ]:
x_star = 1.3247179572  # közelítő
dphi = lambda x: (1/3)*(x+1)**(-2/3)
print(f"phi'(x*) = {dphi(x_star):.6f}  -> elsőrendű konvergencia, q≈{dphi(x_star):.4f}")

phi = lambda x: (x+1)**(1/3)
x = 1.5; errs = []
for _ in range(20):
    x = phi(x); errs.append(abs(x - x_star))
ratios = [errs[i+1]/errs[i] for i in range(len(errs)-1) if errs[i]>1e-12]
print(f'Tapasztalati q: {ratios[-3:]}')

## 14. feladat – Newton-lépések kézzel 📝

$f(x) = x^3 - 2$, $x_0 = 2$.

Végezz **2 Newton-lépést** kézzel!

---
**Megoldás:**

$f'(x) = 3x^2$.

$k=0$: $x_1 = x_0 - \dfrac{f(x_0)}{f'(x_0)} = 2 - \dfrac{8-2}{3\cdot 4} = 2 - \dfrac{6}{12} = 2 - 0.5 = 1.5$

$k=1$: $x_2 = 1.5 - \dfrac{1.5^3-2}{3\cdot 1.5^2} = 1.5 - \dfrac{3.375-2}{6.75} = 1.5 - \dfrac{1.375}{6.75} \approx 1.5 - 0.2037 \approx 1.2963$

In [ ]:
f  = lambda x: x**3 - 2
df = lambda x: 3*x**2
x_star = 2**(1/3)
x = 2.0
for k in range(5):
    xn = x - f(x)/df(x)
    print(f'k={k}: x={x:.8f}, f(x)={f(x):.6f}, x_new={xn:.8f}, hiba={abs(xn-x_star):.2e}')
    x = xn

## 15. feladat – Húrmódszer kézzel 📝

$f(x) = x^2 - 3$, $[a,b]=[1,2]$.

Végezz **2 húrmódszer-lépést** kézzel!

---
**Megoldás:**

$f(1)=-2$, $f(2)=1$. A húrmódszernél $a$ és $b$ **rögzítve** maradnak.

$k=0$: $x_1 = 1 - (-2)\cdot\dfrac{2-1}{1-(-2)} = 1 + \dfrac{2}{3} = \dfrac{5}{3}\approx 1.6667$

$k=1$: $f(x_1)=f(5/3)=25/9-3=-2/9\approx -0.222$
$x_2 = x_1 - f(x_1)\cdot\dfrac{b-a}{f(b)-f(a)} = 5/3 - (-2/9)\cdot\dfrac{1}{3} = 5/3 + 2/27\approx 1.741$

In [ ]:
f = lambda x: x**2 - 3
a, b_ = 1., 2.
fa, fb = f(a), f(b_)
x_star = np.sqrt(3)
print(f'x* = sqrt(3) ≈ {x_star:.6f}')
x = a
for k in range(5):
    xn = x - f(x)*(b_-a)/(fb-fa)
    print(f'k={k}: x={x:.6f}, f(x)={f(x):.6f} -> x_new={xn:.6f}, hiba={abs(xn-x_star):.2e}')
    x = xn

## 16. feladat – Szelőmódszer kézzel 📝

$f(x) = x^2 - 3$, $x_0=1$, $x_1=2$.

Végezz **2 szelőmódszer-lépést** kézzel!

---
**Megoldás:**

$f(x_0)=-2$, $f(x_1)=1$.

$k=1$: $x_2 = x_1 - f(x_1)\cdot\dfrac{x_1-x_0}{f(x_1)-f(x_0)} = 2 - 1\cdot\dfrac{2-1}{1-(-2)} = 2 - \dfrac{1}{3} = \dfrac{5}{3}\approx 1.6667$

$k=2$: $f(x_2)=-2/9$, $x_3 = 5/3 - (-2/9)\cdot\dfrac{5/3-2}{-2/9-1} = 5/3 - (-2/9)\cdot\dfrac{-1/3}{-11/9}$
$= 5/3 - \dfrac{2/27}{11/9} = 5/3 - \dfrac{2}{33} = \dfrac{55-2}{33} = \dfrac{53}{33}\approx 1.6061$

*(A szelőmódszer gyorsabban konvergál, mint a húrmódszer!)*

In [ ]:
f = lambda x: x**2 - 3
x_star = np.sqrt(3)
xs = [1., 2.]
for k in range(6):
    x0, x1 = xs[-2], xs[-1]
    f0, f1 = f(x0), f(x1)
    if abs(f1-f0) < 1e-15: break
    xn = x1 - f1*(x1-x0)/(f1-f0)
    xs.append(xn)
    print(f'k={k+1}: x={xn:.8f}, hiba={abs(xn-x_star):.2e}')

---
# 🐛 Debuggolós feladatok


## 🐛 D1. feladat – Hibás Gauss–Seidel implementáció

Az alábbi kód GS-iterációt próbál megvalósítani, de **nem konvergál** a várt megoldáshoz.
Keresd meg és javítsd a hibát!

*Tipp: Figyelj arra, hogy melyik lépés értékeit használja az $i$-edik egyenlet frissítésekor.*

In [ ]:
# HIBÁS KÓD – javítsd!
def gs_hibas(A, b, x0, n_iter):
    n = len(b)
    x = x0.copy().astype(float)
    for _ in range(n_iter):
        x_new = x.copy()  # <-- ez a gyanús sor
        for i in range(n):
            sigma = sum(A[i,j]*x[j] for j in range(n) if j != i)  # <-- hiba itt!
            x_new[i] = (b[i] - sigma) / A[i,i]
        x = x_new
    return x

A = np.array([[4.,-1],[-1,4]])
b = np.array([3.,7.])
x_star = np.linalg.solve(A, b)
print('Pontos megoldás:', x_star)
print('Hibás GS eredménye:', gs_hibas(A, b, np.zeros(2), 50))
print('(Egyáltalán nem kellene konvergálni GS-sel – de ez csak Jacobi-iteráció!)')

In [ ]:
# JAVÍTOTT KÓD
def gs_javitott(A, b, x0, n_iter):
    n = len(b)
    x = x0.copy().astype(float)
    for _ in range(n_iter):
        # A GS lényege: AZONNAL felhasználjuk a frissen számolt értékeket!
        for i in range(n):
            sigma = sum(A[i,j]*x[j] for j in range(n) if j != i)
            # Figyeld meg: x[j] már az AKTUÁLIS lépés értéke j<i esetén
            # (mert in-place frissítünk)
            x[i] = (b[i] - sigma) / A[i,i]
    return x

print('Javított GS eredménye:', gs_javitott(A, b, np.zeros(2), 30))
print('Pontos megoldás:      ', x_star)

## 🐛 D2. feladat – Hibás Richardson-iteráció

Az alábbi kód Richardson-iterációt valósít meg, de **minden lépésben feleslegesen újraszámolja**
$r = b - Ax$-et a rezidum-frissítés helyett. Javítsd!

*Tipp: A reziduum-frissítési képlet: $r^{(k+1)} = r^{(k)} - A s^{(k)}$*

In [ ]:
# HIBÁS KÓD – javítsd!
def richardson_hibas(A, b, x0, p, n_iter):
    x = x0.copy().astype(float)
    for _ in range(n_iter):
        r = b - A @ x          # <-- ezt nem kellene minden lépésben elvégezni!
        s = p * r
        x = x + s
        # r frissítése hiányzik -> felesleges mátrix-vektor szorzás minden lépésben
    return x

A = np.array([[3.,-1],[-1,3]]); b = np.array([1.,5.])
x_star = np.linalg.solve(A, b)
res = richardson_hibas(A, b, np.zeros(2), 1/3, 50)
print('Hibás (de helyes eredményt ad, csak lassabb):', res)
print('Hiba:', np.linalg.norm(res-x_star))

In [ ]:
# JAVÍTOTT KÓD – hatékony rezidum-frissítéssel
def richardson_javitott(A, b, x0, p, n_iter):
    x = x0.copy().astype(float)
    r = b - A @ x   # csak egyszer számolja ki
    for _ in range(n_iter):
        s = p * r
        x = x + s
        r = r - A @ s   # O(n^2) helyett O(n^2) – de csak EGY mat-vec szorzás!
        # (szemben a hibás verzióval, amely szintén egy szorzást végez,
        # de NEM használja ki a reziduum struktúráját)
    return x

res2 = richardson_javitott(A, b, np.zeros(2), 1/3, 50)
print('Javított eredmény:', res2)
print('Hiba:', np.linalg.norm(res2-x_star))

## 🐛 D3. feladat – Hibás intervallumfelezés

Az alábbi kód intervallumfelezést valósít meg, de **rossz irányba megy** – nem a gyökhöz tart.
Keresd meg a logikai hibát!

In [ ]:
# HIBÁS KÓD – javítsd!
def bisect_hibas(f, a, b, n_iter):
    for _ in range(n_iter):
        c = (a+b)/2
        if f(a)*f(c) < 0:   # <-- HIBÁS feltétel! Mi kellene ide?
            a = c            # <-- ezért a rossz ág marad meg
        else:
            b = c
    return (a+b)/2

f = lambda x: x**3 - x - 1
x_star = 1.3247179572
res = bisect_hibas(f, 1., 2., 20)
print(f'Hibás eredmény: {res:.6f}')
print(f'Pontos gyök:   {x_star:.6f}')
print(f'Hiba: {abs(res-x_star):.6f}  (túl nagy!)')

In [1]:
# JAVÍTOTT KÓD
def bisect_javitott(f, a, b, n_iter):
    for _ in range(n_iter):
        c = (a+b)/2
        if f(a)*f(c) < 0:   # Ha a és c között előjelváltás van...
            b = c            # ...akkor b-t tesszük c-be (a bal ág marad)
        else:
            a = c            # ...különben a-t rakjuk c-be (a jobb ág marad)
    return (a+b)/2

res2 = bisect_javitott(f, 1., 2., 20)
print(f'Javított eredmény: {res2:.8f}')
print(f'Pontos gyök:       {x_star:.8f}')
print(f'Hiba: {abs(res2-x_star):.2e}')

NameError: name 'f' is not defined

## 🐛 D4. feladat – Hibás Newton-módszer

Az alábbi Newton-implementáció **lassú** – csak elsőrendű konvergenciát mutat.
Mi a hiba? Javítsd!

In [ ]:
# HIBÁS KÓD – javítsd!
def newton_hibas(f, df, x0, n_iter):
    x = float(x0)
    df_x0 = df(x0)   # <-- itt a hiba: a deriváltat rögzítve számolja!
    hist = [x]
    for _ in range(n_iter):
        x = x - f(x) / df_x0  # mindig az x0-beli deriváltat használja!
        hist.append(x)
    return x, hist

f  = lambda x: x**2 - 2
df = lambda x: 2*x
x_star = np.sqrt(2)

_, hist_hib = newton_hibas(f, df, 1.0, 15)
errs_hib = np.abs(np.array(hist_hib) - x_star)
print('Hibás Newton hibák:', errs_hib[:8])

In [ ]:
# JAVÍTOTT KÓD
def newton_javitott(f, df, x0, n_iter):
    x = float(x0); hist = [x]
    for _ in range(n_iter):
        x = x - f(x) / df(x)  # minden lépésben az AKTUÁLIS x-ben számolja df-t
        hist.append(x)
    return x, hist

_, hist_jav = newton_javitott(f, df, 1.0, 10)
errs_jav = np.abs(np.array(hist_jav) - x_star)

plt.figure(figsize=(8,4))
plt.semilogy(errs_hib[:15]+[1e-17], 'r-o', markersize=5, label='Hibás (df rögzített)')
plt.semilogy(errs_jav[:10]+[1e-17], 'g-o', markersize=5, label='Javított (valódi Newton)')
plt.title('Newton-módszer: helyes vs hibás implementáció')
plt.xlabel('iteráció'); plt.ylabel('|x_k - √2|'); plt.legend(); plt.grid(True); plt.show()